# Conversational AI - Group 129
**Problem Statement-2:** Study of Embedding Models and Approximate Nearest Neighbor Search - Semantic Quality vs Search Efficiency

**Student Details & Contributions:**
*   **Anirudh Anand - 2025AE05576:** Dataset & embedding preparation and Similarity Matrix comparison.
*   **Member B:** KNN Baseline implementation (Task 4), HNSW vs IVF comparison (Task 5).
*   **Member C:** Evaluation & Analysis, metrics computation, trade-off analysis (Task 6), Qualitative retrieval (Task 7).
*   **Member D:** Visualization & Reporting, final recommendations (Task 8), complete notebook verification and PDF generation.

**Tools & Libraries Used:**
*   `datasets`: To load the Hugging Face text corpus.
*   `sentence-transformers`: To instantiate encoder models and generate embeddings.
*   `faiss-cpu`: For efficient similarity search and dense vector clustering.
*   `numpy` & `pandas`: For vector math and tabular data formatting.
*   `matplotlib`: For performance trade-off visualization.

In [2]:
!pip install datasets sentence-transformers faiss-cpu matplotlib pandas numpy torch


## Module 1

### Task 1: Corpus and Query Dataset Preparation
We selected the `allenai/sciq` dataset from Hugging Face. It contains science exam questions (queries) and their corresponding supporting evidence (corpus passages). 

We iteratively extract valid query-document pairs to guarantee atleast 500 unique queries and record their direct indices. We then pad the corpus with unique supporting passages until it reaches atleast 10,000 documents.

This dataset provides inherent, objective query-document relevance labels, removing subjective bias from our Recall@5 evaluation.

In [4]:
import time
import torch
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

# Load dataset
dataset = load_dataset("allenai/sciq", split="train")

corpus, queries = [], []
relevance_info = {}
seen_supports = set()

# 1. Collect 500 unique queries and their supporting documents
for row in dataset:
    support, question = row['support'].strip(), row['question'].strip()
    if support and support not in seen_supports:
        corpus.append(support)
        seen_supports.add(support)
        queries.append(question)
        relevance_info[len(queries) - 1] = len(corpus) - 1
    if len(queries) == 500:
        break

# 2. Add distractor documents until corpus hits 10,000
for row in dataset:
    support = row['support'].strip()
    if support and support not in seen_supports:
        corpus.append(support)
        seen_supports.add(support)
    if len(corpus) == 10000:
        break

print(f"Corpus size: {len(corpus)} | Query size: {len(queries)}")

Corpus size: 10000 | Query size: 500


### Task 2: Embedding Generation and Pooling
We selected `all-MiniLM-L6-v2` (Mean Pooling) and `multi-qa-mpnet-base-dot-v1` (CLS Pooling). Comparing models with different pooling architectures allows us to analyze how aggregating token weights versus relying on a classification token affects retrieval on our specific Question-Answering dataset. Encoder models are highly appropriate here because their bidirectional self-attention mechanisms compress deep semantic context into dense vectors, enabling search by meaning rather than lexical keyword matching.

In [5]:
metrics = []
models = ['all-MiniLM-L6-v2', 'multi-qa-mpnet-base-dot-v1']
embeddings_dict = {}
query_embs_dict = {}

# Hardcode the pooling strategies since dynamic fetching can fail across different architectures
pooling_strategies = {
    'all-MiniLM-L6-v2': 'Mean Pooling',
    'multi-qa-mpnet-base-dot-v1': 'CLS Pooling'
}

for m_name in models:
    model = SentenceTransformer(m_name)
    
    # Extract architecture details programmatically
    dimension = model.get_embedding_dimension()
    max_seq = model.max_seq_length 
    
    # Fetch the hardcoded pooling strategy
    pooling_mode = pooling_strategies.get(m_name, "Unknown")
        
    # Start the timer
    start = time.time()
    
    # Generate embeddings
    embeddings_dict[m_name] = model.encode(corpus, convert_to_numpy=True)
    query_embs_dict[m_name] = model.encode(queries, convert_to_numpy=True)
    
    # Stop the timer
    duration = time.time() - start
    
    # Calculate Throughput: Total sentences processed divided by total duration
    total_sentences = len(corpus) + len(queries)
    throughput = total_sentences / duration
    
    metrics.append({
        "Model Name": m_name,
        "Dimensions": dimension,
        "Max Inp Length": max_seq,
        "Pooling": pooling_mode,
        "Approx. Time (s)": round(duration, 3),
        "Throughput": round(throughput, 1)
    })

print("Task 2: Embedding Generation Documentation:\n")
print(pd.DataFrame(metrics).to_string(index=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Task 2: Embedding Generation Documentation:

                Model Name  Dimensions  Max Inp Length      Pooling  Approx. Time (s)  Throughput
          all-MiniLM-L6-v2         384             256 Mean Pooling            28.204       372.3
multi-qa-mpnet-base-dot-v1         768             512  CLS Pooling           302.027        34.8


------------------
The benchmark clearly highlights the classic trade-off in embedding models: Speed vs. Capacity. The all-MiniLM-L6-v2 model is built for rapid, resource-constrained environments where high-throughput processing is required. Meanwhile, multi-qa-mpnet-base-dot-v1 is optimized for accuracy in complex Question-Answering (Q&A) and semantic search tasks, trading longer computation times and higher memory usage for deeper semantic understanding.

## Module 2

### Task 3: Similarity Metric Comparison

1. Cosine similarity measures the cosine of the angle between two multi-dimensional vectors in space. It evaluates orientation rather than magnitude:$$\text{Cosine}(u, v) = \frac{u \cdot v}{\Vert{}u\Vert{}_2 \Vert{}v\Vert{}_2} = \frac{\sum_{i=1}^{d} u_i v_i}{\sqrt{\sum_{i=1}^{d} u_i^2} \sqrt{\sum_{i=1}^{d} v_i^2}}$$Range: $[-1, 1]$ (or $[0, 1]$ for non-negative embeddings).Properties: Independent of vector length/magnitude.

2. Dot Product (Inner Product)The dot product computes the sum of the element-wise products of two vectors:$$\text{Dot}(u, v) = u \cdot v = \sum_{i=1}^{d} u_i v_i$$Range: $(-\infty, \infty)$Properties: Combines both vector orientation and magnitude. A document vector with a larger norm ($\Vert{}v\Vert{}_2$) can achieve a higher dot product even if its angular alignment with the query is weaker.

3. L2 / Euclidean DistanceEuclidean distance measures the geometric straight-line distance between two point vectors in $d$-dimensional space:$$\text{L2}(u, v) = \Vert{}u - v\Vert{}_2 = \sqrt{\sum_{i=1}^{d} (u_i - v_i)^2}$$Range: $[0, \infty)$Properties: Measures dissimilarity rather than similarity (smaller values indicate closer/more similar vectors). Sensitive to vector magnitude.

In [6]:
import numpy as np
import pandas as pd
from numpy.linalg import norm
from sentence_transformers import SentenceTransformer

# 1. Load selected model and generate embeddings for sample pairs
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

# Select 1 query and 20 target documents to evaluate ranking across 20 pairs
query_text = queries[0]
doc_texts = corpus[:20]

# Compute unnormalized dense embeddings
query_vec = model.encode(query_text, convert_to_numpy=True)
doc_vecs = model.encode(doc_texts, convert_to_numpy=True)

# Compute L2-normalized embeddings
query_vec_norm = query_vec / norm(query_vec)
doc_vecs_norm = doc_vecs / norm(doc_vecs, axis=1, keepdims=True)

# 2. Compute similarity/distance values across all 20 pairs
results = []
for i in range(20):
    q = query_vec
    d = doc_vecs[i]
    q_n = query_vec_norm
    d_n = doc_vecs_norm[i]
    
    # Raw (Unnormalized) metrics
    cos_sim = np.dot(q, d) / (norm(q) * norm(d))
    dot_raw = np.dot(q, d)
    l2_raw = norm(q - d)
    
    # Normalized metrics
    dot_norm = np.dot(q_n, d_n)
    l2_norm = norm(q_n - d_n)
    
    results.append({
        "Pair ID": f"Q0-D{i}",
        "Doc Preview": doc_texts[i][:35] + "...",
        "Cosine Sim": round(float(cos_sim), 4),
        "Dot (Raw)": round(float(dot_raw), 4),
        "L2 (Raw)": round(float(l2_raw), 4),
        "Dot (Norm)": round(float(dot_norm), 4),
        "L2 (Norm)": round(float(l2_norm), 4)
    })

df_metrics = pd.DataFrame(results)

# 3. Add Document Ranks across metrics (1 = Most relevant)
df_metrics["Rank (Cosine)"] = df_metrics["Cosine Sim"].rank(ascending=False, method="min").astype(int)
df_metrics["Rank (Dot Raw)"] = df_metrics["Dot (Raw)"].rank(ascending=False, method="min").astype(int)
df_metrics["Rank (L2 Raw)"] = df_metrics["L2 (Raw)"].rank(ascending=True, method="min").astype(int)
df_metrics["Rank (Dot Norm)"] = df_metrics["Dot (Norm)"].rank(ascending=False, method="min").astype(int)
df_metrics["Rank (L2 Norm)"] = df_metrics["L2 (Norm)"].rank(ascending=True, method="min").astype(int)

# Display tabular results
print("=== Task 3: 20 Query-Document Pairs Similarity Comparison ===")
display_cols = ["Pair ID", "Cosine Sim", "Dot (Raw)", "L2 (Raw)", "Dot (Norm)", "L2 (Norm)"]
print(df_metrics[display_cols].to_string(index=False))

print("\n=== Document Ranks Across Metrics ===")
rank_cols = ["Pair ID", "Rank (Cosine)", "Rank (Dot Raw)", "Rank (L2 Raw)", "Rank (Dot Norm)", "Rank (L2 Norm)"]
print(df_metrics[rank_cols].to_string(index=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

=== Task 3: 20 Query-Document Pairs Similarity Comparison ===
Pair ID  Cosine Sim  Dot (Raw)  L2 (Raw)  Dot (Norm)  L2 (Norm)
  Q0-D0      0.3220     0.3220    1.1644      0.3220     1.1644
  Q0-D1     -0.0326    -0.0326    1.4371     -0.0326     1.4371
  Q0-D2      0.0346     0.0346    1.3896      0.0346     1.3896
  Q0-D3      0.0624     0.0624    1.3694      0.0624     1.3694
  Q0-D4      0.0707     0.0707    1.3633      0.0707     1.3633
  Q0-D5      0.0835     0.0835    1.3539      0.0835     1.3539
  Q0-D6      0.0252     0.0252    1.3963      0.0252     1.3963
  Q0-D7      0.0171     0.0171    1.4021      0.0171     1.4021
  Q0-D8      0.0475     0.0475    1.3802      0.0475     1.3802
  Q0-D9      0.3721     0.3721    1.1206      0.3721     1.1206
 Q0-D10      0.0125     0.0125    1.4054      0.0125     1.4054
 Q0-D11      0.1312     0.1312    1.3181      0.1312     1.3181
 Q0-D12      0.1289     0.1289    1.3199      0.1289     1.3199
 Q0-D13      0.0068     0.0068    1.4094  


The reason `Cosine Sim`, `Dot (Raw)`, and `Dot (Norm)` are identical (and `L2 (Raw)` equals `L2 (Norm)`) is that **the embeddings generated by `SentenceTransformer` are already L2-normalized unit vectors by default** ($\|u\|_2=1$ and $\|v\|_2=1$).

Because the vectors already have a length (magnitude) of exactly $1$:
1. **Raw Dot Product equals Normalized Dot Product:** Normalizing a vector that already has a magnitude of $1$ does not change its values.
2. **Cosine Similarity equals Dot Product:** The denominator of the cosine formula ($\|u\|_2 \cdot \|v\|_2$) equals $1 \cdot 1 = 1$, making Cosine Similarity mathematically identical to the Dot Product.
3. **L2 Distance is directly derived from Cosine Similarity:** Because the vectors are unit length, the Euclidean distance is a direct mathematical transformation of the dot product ($\sqrt{2-2\cdot\text{dot}}$).

---

#### A. Cosine Similarity vs. Dot Product
The standard formula for Cosine Similarity is:
$$\text{Cosine}(u, v)=\frac{u \cdot v}{\|u\|_2 \|v\|_2}$$

Since the model outputs unit-normalized vectors where $\|u\|_2=1$ and $\|v\|_2=1$:
$$\text{Cosine}(u, v)=\frac{u \cdot v}{1 \cdot 1}=u \cdot v=\text{Dot}(u, v)$$

This is why `Cosine Sim`, `Dot (Raw)`, and `Dot (Norm)` all equal **0.3220** for pair `Q0-D0`.

---

#### B. L2 Distance vs. Dot Product
The squared L2 (Euclidean) distance between two vectors $u$ and $v$ expands as:
$$\|u-v\|_2^2=\|u\|_2^2+\|v\|_2^2-2(u \cdot v)$$

Substituting unit magnitudes ($\|u\|_2=1$ and $\|v\|_2=1$):
$$\|u-v\|_2^2=1+1-2(u \cdot v)=2-2(u \cdot v)$$
$$\|u-v\|_2=\sqrt{2-2(u \cdot v)}$$

* For pair `Q0-D0`, where $u \cdot v = 0.3220$:
  $$\text{L2}=\sqrt{2-2(0.3220)}=\sqrt{1.3560}\approx 1.16447$$
  This matches your table output of **1.1644** perfectly.

---

#### C. Why Document Ranks Do Not Change
* $\text{Cosine}(u, v)$ and $\text{Dot}(u, v)$ are **strictly increasing** functions (higher value = higher rank).
* $\text{L2}(u, v)=\sqrt{2-2\cdot\text{Cosine}(u, v)}$ is a **strictly decreasing** function with respect to Cosine Similarity. As similarity increases, L2 distance decreases.
* Because the transformation is strictly monotonic, **sorting by highest Cosine Similarity, highest Dot Product, or lowest L2 Distance produces the exact same document ordering.**

---


1. **Does ranking change across metrics?** 
   No. When embeddings are L2-normalized, ranking remains strictly invariant across Cosine Similarity, Dot Product, and L2 Distance.
2. **Does normalization affect the results?** 
   Because the underlying model (`all-MiniLM-L6-v2`) outputs pre-normalized unit vectors, applying explicit normalization produces identical results to the raw outputs.
3. **Which metric is most suitable?** 
   **Dot Product on L2-normalized vectors** (or Cosine Similarity) is the most suitable because it reduces distance calculations to simple matrix multiplication ($u \cdot v$), which is computationally faster for vector search engines like FAISS (`IndexFlatIP`).